In [1]:

from pathlib import Path
import json,csv,hashlib
ROOT=Path(r'C:/Users/choud/Desktop/Minor project');TUNE=ROOT/'model_improvement/attention_tuning_v1';DEST=ROOT/'model_improvement/error_diagnosis_v1'
# Definitions only: no trial execution, optimization or test evaluation.
nb=json.loads((TUNE/'01_tuning.ipynb').read_text());exec(compile(''.join(nb['cells'][0]['source']),'existing_definitions_only','exec'))
paths=[TUNE/'tuning_report.md',TUNE/'trial_results.csv',TUNE/'01_tuning.ipynb',TUNE/'02_tuning_evaluation.ipynb',ORIGINAL/'final_attention_report.md',ORIGINAL/'models/attention_3s_best.pt',CR/'models/cnn_3s_best.pt',BR/'models/bilstm_3s_best.pt',TAB,V3/'dataset/eew_dataset_train.h5',V3/'dataset/eew_dataset_validation.h5']
before={str(p):sha(p) for p in paths};rows=list(csv.DictReader((TUNE/'trial_results.csv').open()));ck=torch.load(ORIGINAL/'models/attention_3s_best.pt',map_location='cpu',weights_only=False)
c,b,cc,bc=branches(3);d=temporal('validation',3,c,b);base=encode('validation',3,c,b,cc,bc);head=AttentionHead().to(DEV);head.load_state_dict(ck['head_state']);p,m,_=predict_attention(head,d,ck)
y=d['y'].numpy();actual=d['m'].numpy();reg=d['reg'].numpy();assert np.array_equal(d['ids'],base['ids'])
R={'validation_n':len(y),'earthquakes':int(reg.sum()),'noise':int((~reg).sum()),'thresholds':{str(t):det(y,p,t) for t in [.5,.41]},'models':{},'bands':{},'distributions':{},'family_metadata':{}}
for name,prob,pred in [('Attention',p,m),('CNN',base['cnn_p'],base['cnn_m']),('BiLSTM',base['bilstm_p'],base['bilstm_m'])]:
 R['models'][name]={'detection_05':det(y,prob,.5),'magnitude':magnitude(actual[reg],pred[reg])}
bands=[('M<3',-np.inf,3),('3<=M<4',3,4),('4<=M<5',4,5),('M>=5',5,np.inf)]
for label,lo,hi in bands:
 mask=reg&(actual>=lo)&(actual<hi);err=m[mask]-actual[mask]
 R['bands'][label]={'n':int(mask.sum()),'mae':float(abs(err).mean()),'bias':float(err.mean()),'rmse':float(np.sqrt((err**2).mean())),'underprediction_percent':float((err<0).mean()*100),'fn_05':int((p[mask]<.5).sum()),'fn_041':int((p[mask]<.41).sum())}
for split in ['train','validation']:
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w:
  a=w['magnitude'][:];mask=w['magnitude_mask_3s'][:].astype(bool);a=a[mask];R['distributions'][split]={'eligible_earthquakes':len(a),'mean':float(a.mean()),'std':float(a.std()),'min':float(a.min()),'max':float(a.max()),'counts':{label:int(((a>=lo)&(a<hi)).sum()) for label,lo,hi in bands}}
  # Inspect only metadata names; no family assigned from waveform appearance or IDs.
  R['family_metadata'][split]={'candidate_keys':[k for k in w.keys() if any(word in k.lower() for word in ['family','noise_type','noise_class'])],'candidate_attributes':[k for k in w.attrs if any(word in k.lower() for word in ['family','noise_type','noise_class'])]}
with h5py.File(TAB,'r') as t:R['family_metadata']['tabular_validation_keys']=[k for k in t['validation'].keys() if any(word in k.lower() for word in ['family','noise_type','noise_class'])]
cp,bp=base['cnn_p'],base['bilstm_p'];cm,bm=base['cnn_m'],base['bilstm_m'];ce=(cp>=.5)!=(y==1);be=(bp>=.5)!=(y==1);ec=cm[reg]-actual[reg];eb=bm[reg]-actual[reg]
R['complementarity']={'probability_correlation':float(np.corrcoef(cp,bp)[0,1]),'magnitude_residual_correlation':float(np.corrcoef(ec,eb)[0,1]),'cnn_wrong_bilstm_right':int((ce&~be).sum()),'bilstm_wrong_cnn_right':int((be&~ce).sum()),'both_wrong':int((ce&be).sum()),'both_correct':int((~ce&~be).sum()),'cnn_errors':int(ce.sum()),'bilstm_errors':int(be.sum()),'both_false_positive':int((ce&be&(y==0)).sum()),'both_false_negative':int((ce&be&(y==1)).sum()),'oracle_best_absolute_error_mae':float(np.minimum(abs(ec),abs(eb)).mean()),'cnn_mae':float(abs(ec).mean()),'bilstm_mae':float(abs(eb).mean())}
# Oracle is an unattainable retrospective diagnostic, not a fitted blend or proposed selector.
R['trial_ranges']={k:{'min':min(float(r[k]) for r in rows),'max':max(float(r[k]) for r in rows)} for k in ['val_f1','val_false_positive_rate','val_mae','val_strong_mae','val_strong_underprediction_percent','val_std_ratio']}
R['trial_count']=len(rows);R['original_validation_magnitude']=R['models']['Attention']['magnitude']
print(json.dumps(R,indent=2))


Exact paired inputs; frozen temporal backbone states available.
{
  "validation_n": 2416,
  "earthquakes": 1203,
  "noise": 1213,
  "thresholds": {
    "0.5": {
      "accuracy": 0.7756622516556292,
      "precision": 0.8375893769152196,
      "recall": 0.6816292601828762,
      "f1": 0.7516040329972502,
      "false_positive_rate": 0.13107996702390767,
      "roc_auc": 0.8647048221710083,
      "pr_auc_ap": 0.8869448398872987,
      "confusion_matrix": [
        [
          1054,
          159
        ],
        [
          383,
          820
        ]
      ]
    },
    "0.41": {
      "accuracy": 0.7719370860927153,
      "precision": 0.7767402376910016,
      "recall": 0.7605985037406484,
      "f1": 0.7685846283074339,
      "false_positive_rate": 0.21681780708985984,
      "roc_auc": 0.8647048221710083,
      "pr_auc_ap": 0.8869448398872987,
      "confusion_matrix": [
        [
          950,
          263
        ],
        [
          288,
          915
        ]
      ]
    }

In [2]:

lines=['# 3s EEW error diagnosis','','## Scope and evidence','','Validation-only diagnosis:1,203 eligible earthquakes and1,213 noise. Saved12-trial summaries and relevant evaluation code were read. No per-record validation prediction archive or discarded trial weights was available, so only original frozen Attention/CNN/BiLSTM validation predictions were reproduced. No test inputs or predictions were loaded; no training, fitting, threshold selection, model alteration or repeated preprocessing audit occurred. Train metadata was read for magnitude distributions.','', '## Dominant bottlenecks','','Both detection discrimination and magnitude compression constrain improvement; no single causal mechanism is established. Detection F1 failed to improve across all12 trial operating points, while overall MAE moved by only a few thousandths and strong-event MAE worsened for every trial. Therefore this is not simply a successful magnitude improvement rejected by the detection objective. Loss-weight conflict remains a hypothesis, not a measured gradient result.','', '| Trial | Reg weight | Variant | F1 | FPR | MAE | Strong MAE | Strong underprediction % | SD ratio |','|---|---:|---|---:|---:|---:|---:|---:|---:|']
for row in rows:lines.append(f"| {row['trial']} | {row['reg_weight']} | {row['variant']} | "+' | '.join(f'{float(row[k]):.6f}' for k in ['val_f1','val_false_positive_rate','val_mae','val_strong_mae','val_strong_underprediction_percent','val_std_ratio'])+' |')
lines+=['','Original validation operating point: F1=0.768585, FPR=0.216818, overall MAE=0.615548, strong MAE=1.043461, strong underprediction85.50%, predicted/actual SD=0.605680. Trial F1 spans0.765403–0.767836. Best MAE0.613029 improves by only0.002518; strong MAE spans1.061949–1.098798, uniformly worse. Minimum FPR0.214345 improves by only0.002473, far short of the declared0.02 reduction. None met the original selection criteria; those criteria were not changed here.','', 'Trials10/11/12 provide the closest controlled loss comparison (same LR, width, dropout, regression weight, seed; checkpoint epochs differ): standard strong MAE1.086542; strong-bin weighting1.098798; asymmetric penalty1.077363. The asymmetric variant improves relative to trial10 but remains worse than the original1.043461. Trial11 also increases overall MAE to0.616033. This supports a local bias/error tradeoff, not evidence that weighting solved compression. Other cross-trial effects are confounded by simultaneous settings changes. Unweighted BCE+Huber checkpoint selection can fail to favor tail-specific improvements; epochwise strong-event predictions were not saved, so this is a plausible selection-objective limitation, not a demonstrated cause.','', '## False positives and false negatives','','| Threshold | TN | FP | FN | TP | Precision | Recall | F1 | FPR |','|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for t,q in R['thresholds'].items():
 (tn,fp),(fn,tp)=q['confusion_matrix'];lines.append(f'| {t} | {tn} | {fp} | {fn} | {tp} | '+' | '.join(f'{q[k]:.6f}' for k in ['precision','recall','f1','false_positive_rate'])+' |')
lines+=['','Changing0.50 to0.41 adds95 true positives and104 false positives. Recall rises7.90 percentage points, but FPR rises8.57 points. Threshold adjustment trades errors rather than creating new discriminative information. The0.41 point meets the study’s FPR<=30%/recall>=70% research constraint; it still misclassifies263/1,213 noise windows and misses288/1,203 earthquakes. Operational acceptability is UNKNOWN without alarm-cost, prevalence and continuous-stream false-alarm requirements. Window FPR cannot be converted to false alarms per hour from these records.','', '## Error by true magnitude (eligible validation earthquakes)','','| Range | N | MAE | RMSE | Signed error | Underprediction % | FN at0.50 | FN at0.41 |','|---|---:|---:|---:|---:|---:|---:|---:|']
for label,v in R['bands'].items():lines.append(f"| {label} | {v['n']} | {v['mae']:.6f} | {v['rmse']:.6f} | {v['bias']:.6f} | {v['underprediction_percent']:.3f} | {v['fn_05']} | {v['fn_041']} |")
lines+=['','All191 M<3 predictions overestimate magnitude; mean error+0.9657. M>=5 has mean error−0.9457 and85.50% underprediction, despite only7/131 false negatives at0.41. Strong-earthquake detection therefore does not imply accurate strong-event magnitude estimation. Global mean error+0.0032 hides cancellation between opposite tail biases. Predicted variance0.30728 versus true0.83762, SD ratio0.60568 and regression slope0.34575 quantify compression. Overall RMSE0.75213 and R2=0.32463 show substantial unexplained variation. This pattern is consistent with central-value shrinkage, but does not prove whether loss, frozen representation or limited synthetic observability causes it.','', '## Magnitude distributions','','| Split | Eligible EQ | Mean | SD | M<3 | 3–4 | 4–5 | M>=5 | Strong fraction |','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for split,v in R['distributions'].items():lines.append(f"| {split} | {v['eligible_earthquakes']} | {v['mean']:.6f} | {v['std']:.6f} | "+' | '.join(str(v['counts'][label]) for label,_,_ in bands)+f" | {v['counts']['M>=5']/v['eligible_earthquakes']:.4%} |")
lines+=['','Train/validation means, spreads and strong-event proportions closely match; coarse magnitude-distribution shift is not supported as the main cause. Strong events constitute only about10.9% in each split, making average loss dominated by other events. Minority representation is evidence; a causal claim that imbalance alone explains underprediction is unsupported. The tested weighting variants did not improve strong errors over the original. Coarse counts do not rule out shifts in distance, station, signal-to-noise or other unmeasured conditions.','', '## CNN–BiLSTM complementarity','','Metrics below use both standalone models at0.50 on the same validation records. Different standalone calibration can affect classification overlap.','', '```json',json.dumps(R['complementarity'],indent=2),'```','', 'There is some detection complementarity:244 CNN errors are correctly classified by BiLSTM, while413 BiLSTM errors are correctly classified by CNN;306 errors are shared, including232 shared misses. This does not prove a learned fusion can identify which branch is correct. Probability correlation0.800 is substantial, while magnitude residual correlation0.983 indicates strongly shared magnitude errors. Oracle minimum-absolute-error MAE0.568785 is an unattainable target-aware selector diagnostic, not an achievable model or validated improvement. No blending coefficients or gates were fitted. These results do not justify more blind fusion-head search; they also do not prove that all fusion strategies are impossible.','', '## Frozen representations and unavailable causes','','All trials adapted attention/heads atop the same frozen extractors. Near-flat validation performance across widths64/128, learning rates and loss weights is compatible with a representation bottleneck. Both branches derive from the same synthetic observations, consistent with correlated residuals. However, no controlled unfreezing comparison, representation sufficiency test, gradient-conflict measurement or irreducible-error analysis exists. Thus “frozen backbones caused failure” is SPECULATION, not a finding. Several selected epochs are near the35-epoch cap; convergence beyond the cap was not measured, so optimal convergence cannot be claimed.','', 'Noise-family shortcuts: UNKNOWN. No explicit family/noise-type keys or attributes were found in the inspected train/validation waveform schema or validation feature group. No labels were inferred from IDs, amplitudes or waveform appearance; missing trustworthy metadata prevents family-specific false-positive rates. This does not prove such metadata is absent elsewhere.','', 'Unavailable: per-record predictions for discarded trials, trial-specific magnitude-band/FN/FP overlap, epochwise strong-event selection diagnostics, trustworthy noise-family labels, multiple-seed variability and operational false-alarm rates. Trial CSV gives aggregate fixed-.5 and selected-threshold metrics, not arbitrary0.41 per-trial confusion matrices. No missing measurements were invented.','', '## Recommendation','','**NO FURTHER TUNING at this stage.** The available evidence does not identify one sufficiently supported intervention:12 head/attention trials failed, both magnitude-loss variants underperformed the original on strong events, and shared regression errors are very high. Keep the original model and its documented operating tradeoff. Do not infer that another loss, more fusion capacity or backbone unfreezing will help without a causal diagnostic. No new experiment is recommended or initiated here.','', 'Synthetic oracle-P alignment, shared signal source, single seed and reused validation set limit this diagnosis. These descriptive validation findings are not an independent confirmation or real-world EEW claim.','', '## Source preservation','']
changed=[p for p,v in before.items() if sha(Path(p))!=v];assert not changed
lines.append(f'All {len(before)} hashed source dependencies unchanged. Only diagnostic_analysis.ipynb and diagnostic_report.md created; no checkpoints or data written.')
for path,digest in before.items():lines.append(f'- `{path}`: `{digest}`')
with (DEST/'diagnostic_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Diagnostic report complete; no further tuning recommended; original files unchanged.')


Diagnostic report complete; no further tuning recommended; original files unchanged.
